In [56]:
'''
python3 -m venv venv
source venv/bin/activate
pip install pandas numpy scikit-learn torch transformers biopython
'''

import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, max_error, mean_absolute_error
import torch
from transformers import AutoTokenizer, AutoModel

In [46]:
df = pd.read_csv('pHopt_data.csv')
print(df.head())

   Unnamed: 0   Accession                       Organism  EC Number  \
0           1      P21673                   Homo sapiens   2.3.1.57   
1           2      I3WU34     Exiguobacterium acetylicum  3.2.1.142   
2           3  A0A0B5JT51            Exiguobacterium sp.   3.2.1.41   
3           4  A0A1V0FWX7  Geobacillus thermocatenulatus   3.2.1.41   
4           5      P52902                  Pisum sativum  1.2.1.104   

   Sample Weight  pHopt     Split Test <20% to Train  \
0          0.366   7.47  Training                NaN   
1          0.366   6.00  Training                NaN   
2          0.366   8.50  Training                NaN   
3          0.366   6.50  Training                NaN   
4          0.366   7.60  Training                NaN   

                                            Sequence  
0  MAKFVIRPATAADCSDILRLIKELAKYEYMEEQVILTEKDLLEDGF...  
1  MKRIRSVCIVALTFALIFSGLSLSGQALEKGKSTLVIIHYKEDPNA...  
2  MNRLKSVCAVVLTFALIFSLFPVNSLALEKGKSTLVIIHYKEDKTS...  
3  MLHISRTFAAYLD

In [57]:
# ESM-2 
model_name = "facebook/esm2_t30_150M_UR50D" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
esm_model = AutoModel.from_pretrained(model_name)

def get_esm_embedding(sequence):
    """
    převed protein seq na vektor pomocí ESM-2
    """
    inputs = tokenizer(sequence, return_tensors="pt", add_special_tokens=True)     # tokenizace
    
    with torch.no_grad():           # vypnu gradienty model netrénujeme
        outputs = esm_model(**inputs)
    
    last_hidden_states = outputs.last_hidden_state   #skrytý stavy - vektory pro aa
    sequence_embedding = last_hidden_states.mean(dim=1).squeeze().numpy()     # mean Pooling - 1D vektor    
    return sequence_embedding

Loading weights: 100%|██████████| 486/486 [00:00<00:00, 6471.62it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [58]:
# seq -> vektor
X_embeddings = df['Sequence'].apply(get_esm_embedding)

# 2D matice
X_features_esm = pd.DataFrame(X_embeddings.tolist())
features_esm = list(X_features_esm.columns) 
df_model_esm = pd.concat([X_features_esm, df[['pHopt', 'Split']]], axis=1)

In [59]:
'''def calculate_sample_weights(y_values):
     #'bin-inverse' ze studie
    y = np.array(y_values)
    
    acidic_mask = y <= 5
    neutral_mask = (y > 5) & (y < 9)
    alkaline_mask = y >= 9
    
    w_acidic = 1.0 / np.sum(acidic_mask) if np.sum(acidic_mask) > 0 else 0
    w_neutral = 1.0 / np.sum(neutral_mask) if np.sum(neutral_mask) > 0 else 0
    w_alkaline = 1.0 / np.sum(alkaline_mask) if np.sum(alkaline_mask) > 0 else 0
    
    weights = np.zeros(len(y))
    weights[acidic_mask] = w_acidic
    weights[neutral_mask] = w_neutral
    weights[alkaline_mask] = w_alkaline
    
    weights = weights / np.mean(weights)
    return weights
'''

In [60]:
train_data_esm = df_model_esm[df_model_esm['Split'] == 'Training']
test_data_esm = df_model_esm[df_model_esm['Split'] == 'Testing']

X_test_esm = test_data_esm[features_esm]
y_test_esm = test_data_esm['pHopt']

X_train_esm = train_data_esm[features_esm]
y_train_esm = train_data_esm['pHopt']

#train_weights = calculate_sample_weights(y_train_esm)

In [64]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('svr', SVR(kernel='rbf', gamma='scale'))
])

param_grid_svr = {
    'pca__n_components': [100, 150, 200, 250, 300, 350],
    'svr__C': [0.5, 1, 2],
    'svr__epsilon': [0.2, 0.5, 0.8]
}

grid_search_svr_esm = GridSearchCV(
    estimator=pipeline, 
    param_grid=param_grid_svr, 
    cv=5, 
    scoring='neg_mean_squared_error', 
    n_jobs=-1
)

grid_search_svr_esm.fit(X_train_esm, y_train_esm, svr__sample_weight=train_weights)

best_svr_esm = grid_search_svr_esm.best_estimator_

In [65]:
y_pred_esm = best_svr_esm.predict(X_test_esm)

mse_esm = mean_squared_error(y_test_esm, y_pred_esm)
rmse = np.sqrt(mse_esm)
mae = mean_absolute_error(y_test_esm, y_pred_esm)
max_err = max_error(y_test_esm, y_pred_esm)

r2_esm = r2_score(y_test_esm, y_pred_esm)
r2_train = r2_score(y_train_esm, best_svr_esm.predict(X_train_esm))


print(f"Best params SVR : {grid_search_svr_esm.best_params_}")
print(f"MSE: {mse_esm:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"Max error: {max_err:.3f}")
print(f"R^2: {r2_esm:.3f}")
print(f"R^2 trénovacích dat: {r2_train:.3f}")



Best params SVR : {'pca__n_components': 350, 'svr__C': 2, 'svr__epsilon': 0.8}
MSE: 0.811
RMSE: 0.901
MAE: 0.685
Max error: 4.621
R^2: 0.391
R^2 trénovacích dat: 0.620


In [53]:
sekvence_lig = "MNTQRKYGRTWHYPFSPGTTSDDRINTDYWQDLKAITQLVHTEKLDGENNCLNRYGVFARSHATPTQSAWTYKIRQRWQLLKNDLGDLELFGENLYAVHSIEYRALEQDFYLFAVRCQDMWLSWEEVQFYAALFDFPCVPEISGPQPGNDEKSWQRDFLALANARGTFDPWDTQTWQPCTLEGIVSRNHDAFSVADFSHNVFKYVRKNHVKTTVHWKRHWQRARMAHEFVYGEQS"

nove_priznaky_dict = get_esm_embedding(sekvence_lig)

nova_data_df = pd.DataFrame([nove_priznaky_dict], columns=features_esm)
odhadnute_pH = best_svr_esm.predict(nova_data_df)

print(f"Predikované optimální pH pro ligasu je: {odhadnute_pH[0]:.2f}")

Predikované optimální pH pro ligasu je: 7.25


In [54]:
sekvence_ntp = "MIWQLTDDKRWSALRQRFSWVEEMHHTPQDPEHHGEGDVGVHTEMVLNALITLPEFQQLPAQQQEVLWAAVLLHDVEKRSTTVQENGRIQSPGHARRGELTARQILWRDIPTPFVLREQIVALVRLHGLPLWLLERPEPERLLLTAAMRIDTRLLALLARADLLGRQSPDQQSMLERIDLFELFCHEQQCWGKMRPFVSDSARWHYLTHEQSSPDFVPWGAEPFEVILLCGLPGMGKDRYISEQCQGIDVISLDDMRRRINASPDDKTATGRIVQQAKEEARVFLRQKKPFIWNATNITRQLRSQLISLFTAYGARVKIVYLEVPWAQWKQQNARREYAVPEAVVMRMASRLEVPQPDEAHSVEYRVIER"


nove_priznaky_dict = get_esm_embedding(sekvence_ntp)

nova_data_df = pd.DataFrame([nove_priznaky_dict], columns=features_esm)
odhadnute_pH = best_svr_esm.predict(nova_data_df)

print(f"Predikované optimální pH pro atpasu je: {odhadnute_pH[0]:.2f}")

Predikované optimální pH pro atpasu je: 7.77


In [55]:
sekvence_operon = "MNTQRKYGRTWHYPFSPGTTSDDRINTDYWQDLKAITQLVHTEKLDGENNCLNRYGVFARSHATPTQSAWTYKIRQRWQLLKNDLGDLELFGENLYAVHSIEYRALEQDFYLFAVRCQDMWLSWEEVQFYAALFDFPCVPEISGPQPGNDEKSWQRDFLALANARGTFDPWDTQTWQPCTLEGIVSRNHDAFSVADFSHNVFKYVRKNHVKTTVHWKRHWQRARMAHEFVYGEQSMIWQLTDDKRWSALRQRFSWVEEMHHTPQDPEHHGEGDVGVHTEMVLNALITLPEFQQLPAQQQEVLWAAVLLHDVEKRSTTVQENGRIQSPGHARRGELTARQILWRDIPTPFVLREQIVALVRLHGLPLWLLERPEPERLLLTAAMRIDTRLLALLARADLLGRQSPDQQSMLERIDLFELFCHEQQCWGKMRPFVSDSARWHYLTHEQSSPDFVPWGAEPFEVILLCGLPGMGKDRYISEQCQGIDVISLDDMRRRINASPDDKTATGRIVQQAKEEARVFLRQKKPFIWNATNITRQLRSQLISLFTAYGARVKIVYLEVPWAQWKQQNARREYAVPEAVVMRMASRLEVPQPDEAHSVEYRVIER"


nove_priznaky_dict = get_esm_embedding(sekvence_ntp)

nova_data_df = pd.DataFrame([nove_priznaky_dict], columns=features_esm)
odhadnute_pH = best_svr_esm.predict(nova_data_df)

print(f"Predikované optimální pH pro operon je: {odhadnute_pH[0]:.2f}")

Predikované optimální pH pro operon je: 7.77
